<a href="https://colab.research.google.com/github/justorfc/Est_Python_R_2026_1/blob/main/12_Semana_12_Sesion_02_Aplicacion_GLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Llegamos a la gran conclusión del contenido analítico del curso.

Esta sesión es la joya de la corona, ya que le otorga a los estudiantes de ingeniería la capacidad de modelar escenarios altamente realistas que la regresión lineal clásica simplemente no puede procesar.

### 📘 Estructura del Notebook: Semana 12 - Sesión 2

**Título del Notebook:** `Semana_12_Sesion_02_Aplicacion_GLM.ipynb`

#### **Celda 1: Markdown (Encabezado y Fase 1: IA)**


# SEMANA 12: Modelos No Lineales y GLM
**Asignatura:** Estadística Aplicada con Python y R
**Sesión 2:** Aplicación Práctica de Regresión Logística y Poisson.

## Fase 1: Actividad "Estudia y Aprende" (Aplicación)
Antes de ajustar nuestros Modelos Lineales Generalizados (GLM), pídele a tu IA que te explique cómo interpretar los resultados, ya que la lectura de estos coeficientes cambia drásticamente respecto a lo que aprendimos en la Semana 9.

**PROMPT 2 - APLICACIÓN:**
> Actúa como tutor experto en GLM con Python y R.
> 1. Muéstrame cómo ajustar regresión logística.
> 2. Cómo interpretar coeficientes.
> 3. Cómo ajustar regresión Poisson.
> 4. Cómo interpretar tasa esperada.
> 5. Explica diferencias con regresión lineal.
> Interpreta resultados como lo haría un ingeniero.
> No solo muestres código; explica qué significa cada resultado.

#### **Celda 2: Markdown (Fase 2: Demostración Docente - Contexto)**

## Fase 2: Demostración Docente en Python
Ajustaremos dos de los modelos GLM más utilizados en ingeniería usando la librería `statsmodels`:

1. **Regresión Logística (Logit):** Para predecir variables de respuesta binarias ($Y \in \{0, 1\}$). Ejemplo: Dado un número de horas de uso, ¿cuál es la probabilidad de que una máquina falle? La interpretación se hace a través de los *Odds Ratios* (Razón de probabilidades).
2. **Regresión Poisson:** Para predecir conteos puros ($Y \in \{0, 1, 2, \dots\}$). Ejemplo: ¿Cuántas grietas se esperan en un kilómetro de pavimento sometido a cierto tráfico vehicular pesado? La interpretación se centra en la tasa esperada ($\lambda$).

#### **Celda 3: Código Python (Ajuste e Interpretación: Regresión Logística)**

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
np.random.seed(123)

# ==========================================
# 1. REGRESIÓN LOGÍSTICA (Variable Binaria)
# ==========================================
# Contexto: Horas de uso de un motor vs Estado (0=Funciona, 1=Falla)
horas_uso = np.random.uniform(100, 1000, 100)
# Ecuación lineal subyacente (Logit)
logit_p = -5.0 + 0.01 * horas_uso
# Transformación Sigmoide
prob_falla = 1 / (1 + np.exp(-logit_p))
estado_falla = np.random.binomial(1, prob_falla)

df_log = pd.DataFrame({'Horas': horas_uso, 'Falla': estado_falla})

# Ajuste del Modelo Logístico usando sm.Logit
X_log = sm.add_constant(df_log['Horas'])
Y_log = df_log['Falla']
modelo_logit = sm.Logit(Y_log, X_log).fit(disp=0) # disp=0 oculta el texto de iteración

print("--- RESULTADOS REGRESIÓN LOGÍSTICA ---")
print(modelo_logit.summary().tables[1])

# Interpretación del Coeficiente (Odds Ratio)
coef_horas = modelo_logit.params['Horas']
odds_ratio = np.exp(coef_horas)
print(f"\nCoeficiente (Horas): {coef_horas:.4f}")
print(f"Odds Ratio: {odds_ratio:.4f}")

# Interpretación Ingenieril:
# En regresión logística no decimos "por cada hora la probabilidad sube X".
# Decimos: "Por cada hora extra de uso, las 'chances' (odds) de que el motor falle
# se multiplican por {odds_ratio:.4f}." Como es mayor a 1, el riesgo aumenta con el tiempo.

#### **Celda 4: Código Python (Ajuste e Interpretación: Regresión Poisson)**

In [ ]:
# ==========================================
# 2. REGRESIÓN POISSON (Conteos)
# ==========================================
# Contexto: Miles de vehículos pesados por día vs Número de grietas en pavimento (Ing. Civil)
trafico_miles = np.random.uniform(5, 25, 100)
# Ecuación lineal con enlace logarítmico
log_lambda = 0.5 + 0.08 * trafico_miles
# La tasa esperada (lambda) es la exponencial del modelo lineal
lambda_esperado = np.exp(log_lambda)
num_grietas = np.random.poisson(lambda_esperado)

df_pois = pd.DataFrame({'Trafico': trafico_miles, 'Grietas': num_grietas})

# Ajuste del Modelo Poisson usando sm.GLM y la familia Poisson
X_pois = sm.add_constant(df_pois['Trafico'])
Y_pois = df_pois['Grietas']
modelo_poisson = sm.GLM(Y_pois, X_pois, family=sm.families.Poisson()).fit()

print("\n--- RESULTADOS REGRESIÓN POISSON ---")
print(modelo_poisson.summary().tables[1])

# Interpretación de la Tasa
coef_trafico = modelo_poisson.params['Trafico']
efecto_multiplicativo = np.exp(coef_trafico)
print(f"\nEfecto Multiplicativo sobre la Tasa Esperada: {efecto_multiplicativo:.4f}")

# Interpretación Ingenieril:
# En Poisson, el coeficiente nos da un cambio porcentual.
# "Por cada mil vehículos pesados adicionales diarios, el número esperado de grietas
# se multiplica por {efecto_multiplicativo:.4f} (es decir, aumenta aproximadamente un {(efecto_multiplicativo-1)*100:.1f}%)."

#### **Celda 5: Markdown (Transición a R y RMarkdown)**

## Trabajo Autónomo: Transición a R y RMarkdown

R unifica el ajuste de todos los Modelos Lineales Generalizados bajo la misma y elegante función: `glm()`.

**Paso 1: Validación en Colab (Entorno R)**
1. Crea un nuevo Notebook en Colab y cambia el entorno a **R**.
2. Ejecuta el siguiente código para ajustar ambos modelos especificando su `family`.

```R
# Recrear los datos en R
set.seed(123)
# Datos Logísticos
horas <- runif(100, 100, 1000)
prob <- 1 / (1 + exp(-(-5.0 + 0.01 * horas)))
falla <- rbinom(100, 1, prob)
datos_log <- data.frame(Horas=horas, Falla=falla)

# Datos Poisson
trafico <- runif(100, 5, 25)
lambda_esp <- exp(0.5 + 0.08 * trafico)
grietas <- rpois(100, lambda_esp)
datos_pois <- data.frame(Trafico=trafico, Grietas=grietas)

# 1. Regresión Logística (family = binomial)
print("--- MODELO LOGÍSTICO ---")
modelo_r_log <- glm(Falla ~ Horas, family=binomial(link="logit"), data=datos_log)
print(summary(modelo_r_log)$coefficients)

# 2. Regresión Poisson (family = poisson)
print("--- MODELO POISSON ---")
modelo_r_pois <- glm(Grietas ~ Trafico, family=poisson(link="log"), data=datos_pois)
print(summary(modelo_r_pois)$coefficients)

```

**Paso 2: Documentación en Posit Cloud y Despliegue en RPubs**

1. Abre tu espacio en **Posit Cloud** y crea tu último documento **RMarkdown**.
2. Transfiere el código de R.
3. Escribe tu análisis final del curso: *Justifica teóricamente por qué utilizar una regresión lineal clásica para modelar el número de grietas en una carretera podría resultar en la predicción matemática de un "número negativo de grietas". Explica cómo el modelo Poisson previene este absurdo físico.*
4. Compila (Knit) a HTML y publica en **RPubs**.

#### **Celda 6: Markdown (CIERRE DEL TEMA Y HOJA EVALUABLE)**

## CIERRE DEL TEMA: Generación de Resumen Guía

¡Felicidades! Has completado el bloque de modelado analítico avanzado. Usa tu IA por última vez en esta unidad para generar tu resumen de los Modelos Lineales Generalizados.

**PROMPT DE CIERRE GLOBAL:**
> Genera un resumen estructurado para escribir a mano en UNA sola hoja.
> Formato obligatorio:
> A) Idea central (1-2 líneas).
> B) 6-10 viñetas organizadas lógicamente.
> C) 3 relaciones clave (por qué/cómo).
> D) 1 ejemplo aplicado a ingeniería.
> E) 3 preguntas de autoevaluación con respuesta.
> F) Cierre: "Hoy aprendí que ..."

⚠️ **RECUERDA (EVIDENCIA EVALUABLE):**
En el reverso de tu hoja física titulada *"Modelos No Lineales y GLM en Ingeniería"*, escribe a mano:
* La idea central de por qué necesitamos GLMs.
* Las diferencias entre variables continuas, binarias y de conteo.
* Qué modela la Regresión Logística y qué es el enlace Logit.
* Qué modela la Regresión Poisson y qué es el enlace Logarítmico.
* La interpretación ingenieril de los coeficientes en estos modelos.
* Un ejemplo aplicado.
* Las 3 preguntas con sus respuestas y tu reflexión personal final.

¡Hemos completado la estructuración de todos los Notebooks solicitados desde la Semana 2 hasta la Semana 12! Ha sido un trabajo extenso pero el resultado es un material de calidad excepcional para los estudiantes.